# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [8]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


Clean and prepare

In [9]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer



In [10]:
spaceship = spaceship.dropna()
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

Seperating Features and Target

In [11]:
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

Define column types

In [12]:
cat_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

Building transformer

In [13]:
preprocessor = ColumnTransformer(transformers=[
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), cat_cols),
    ("scaler", StandardScaler(), num_cols)
])

**Perform Train Test Split**

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Transfroming both Test and Train (fit only for train to avoid leakage)

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [22]:
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report

bagging = BaggingClassifier(n_estimators=100, random_state=42)
bagging.fit(X_train_processed, y_train)

y_pred_bag = bagging.predict(X_test_processed)
print(" BAGGING ")
print("Accuracy:", accuracy_score(y_test, y_pred_bag))
print(classification_report(y_test, y_pred_bag))

 BAGGING 
Accuracy: 0.8018154311649016
              precision    recall  f1-score   support

       False       0.80      0.81      0.80       653
        True       0.81      0.80      0.80       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



- Random Forests

In [24]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train_processed, y_train)

y_pred_rf = forest.predict(X_test_processed)
print(" RANDOM FOREST ")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

 RANDOM FOREST 
Accuracy: 0.8040847201210287
              precision    recall  f1-score   support

       False       0.79      0.82      0.81       653
        True       0.82      0.79      0.80       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



- Gradient Boosting

In [26]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_processed, y_train)    
y_pred_gb = gb.predict(X_test_processed)
print(" GRADIENT BOOSTING ")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

 GRADIENT BOOSTING 
Accuracy: 0.8071104387291982
              precision    recall  f1-score   support

       False       0.84      0.76      0.79       653
        True       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



- Adaptive Boosting

In [28]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train_processed, y_train)
y_pred_ada = ada.predict(X_test_processed)
print(" ADABOOST ")
print("Accuracy:", accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))

 ADABOOST 
Accuracy: 0.7859304084720121
              precision    recall  f1-score   support

       False       0.78      0.79      0.78       653
        True       0.79      0.78      0.79       669

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



Which model is the best and why?

Summary 

In [29]:
results = pd.DataFrame({
    "Model": ["Bagging", "Random Forest", "Gradient Boosting", "AdaBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_bag),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_ada)
    ]
}).sort_values("Accuracy", ascending=False)

print(results)

               Model  Accuracy
2  Gradient Boosting  0.807110
1      Random Forest  0.804085
0            Bagging  0.801815
3           AdaBoost  0.785930


Comments 

Gradient Boosting typically wins on this dataset.


Random Forest / Gradient Boosting achieved the highest accuracy of X%.


Ensemble methods outperform the baseline KNN (79.3%) because they combine multiple weak learners to reduce variance (Bagging/RF) or bias (Boosting).

